In [9]:
# ═══════════════════════════════════════════════
# DR Congo PEP → 제재 대조 → 제재 인물 뉴스 모니터
# ═══════════════════════════════════════════════
import sys, subprocess, importlib
try:
    import feedparser
except ImportError:                                   # 없으면 현재 환경에 자동 설치
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "feedparser"])
    importlib.invalidate_caches()
    import feedparser

import os, re, time, unicodedata
from collections import defaultdict
from datetime import datetime
from urllib.parse import urljoin, quote
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.opensanctions.org"
HEADERS = {"User-Agent": "Mozilla/5.0 (study project)"}
COUNTRY = "DR Congo"
PREFIX = "drc"
EXPECTED = 561

LOOP = False                  # True: 30분마다 자동 업데이트 / False: 한 번만 실행
INTERVAL = 30 * 60
WATCH_EXTRA = []              # 감시에 직접 추가할 이름 (예: ["Felix Tshisekedi"])


def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))


def tokens(s):
    return [t for t in re.split(r"[\s\-,.']+", strip_accents(s).lower()) if len(t) > 1]


# ───────────────────────────────────────────────
# 1단계: /pep/ 에서 DR Congo 데이터셋 찾기 → CSV 받기
# ───────────────────────────────────────────────
res = requests.get(f"{BASE}/pep/", headers=HEADERS, timeout=10)
print("상태 코드:", res.status_code)
if res.status_code != 200 or COUNTRY not in res.text:
    raise SystemExit("페이지를 못 받았거나 국가 이름이 없어요.")

soup = BeautifulSoup(res.text, "html.parser")
sources = {}
for a in soup.find_all("a", href=True):
    href, text = a["href"], a.get_text(strip=True)
    if href.startswith("/datasets/") and COUNTRY in text:
        sources[href.strip("/").split("/")[-1]] = text

print(f"\n{COUNTRY} 데이터셋 {len(sources)}개")
for slug, name in sources.items():
    print(f" - {name} ({slug})")
if not sources:
    raise SystemExit("데이터셋 링크를 못 찾았어요.")

frames = []
for slug, name in sources.items():
    time.sleep(10)                                   # robots.txt Crawl-delay 준수
    ds_url = f"{BASE}/datasets/{slug}/"
    ds_soup = BeautifulSoup(requests.get(ds_url, headers=HEADERS, timeout=10).text, "html.parser")
    links = [urljoin(ds_url, a["href"]) for a in ds_soup.find_all("a", href=True) if ".csv" in a["href"]]
    csv_url = links[0] if links else f"https://data.opensanctions.org/datasets/latest/{slug}/targets.simple.csv"
    df_src = pd.read_csv(csv_url)
    df_src["source"] = name
    frames.append(df_src)
    print(f"✅ {name}: {len(df_src)}행")

peps = pd.concat(frames, ignore_index=True)
if "schema" in peps:
    peps = peps[peps["schema"] == "Person"]
peps = peps.drop_duplicates("id").reset_index(drop=True)
print(f"\n인물 {len(peps)}명 (예상 약 {EXPECTED}명)")

# ───────────────────────────────────────────────
# 2단계: 제재명단 대조 (ID 확정 + 어순 무관 이름·별칭 후보)
# ───────────────────────────────────────────────
SANC_URL = "https://data.opensanctions.org/datasets/latest/sanctions/targets.simple.csv"
SANC_FILE = "sanctions_cache_v2.csv"
keep = {"id", "schema", "name", "aliases", "birth_date", "countries", "sanctions", "dataset"}

if os.path.exists(SANC_FILE):
    sanc = pd.read_csv(SANC_FILE, low_memory=False)
else:
    print("\n제재명단 다운로드 중... (처음 한 번만)")
    sanc = pd.read_csv(SANC_URL, usecols=lambda c: c in keep, low_memory=False)
    sanc.to_csv(SANC_FILE, index=False)
print(f"제재명단 {len(sanc):,}건")

id_hits = peps.merge(sanc, on="id", how="inner", suffixes=("", "_sanc"))

index = defaultdict(list)
for r in sanc[sanc["schema"] == "Person"].to_dict("records"):
    names = [r["name"]] + (str(r["aliases"]).split(";") if pd.notna(r.get("aliases")) else [])
    for n in names:
        tk = frozenset(tokens(n))
        if len(tk) >= 2:
            for t in tk:
                index[t].append((tk, r["id"], n, r.get("birth_date"), r.get("dataset")))

rows = []
for p in peps.to_dict("records"):
    ptk, seen_ids = set(tokens(p["name"])), set()
    pbd = str(p.get("birth_date"))[:10]
    for t in ptk:
        for tk, sid, sname, sbd, sds in index.get(t, []):
            if sid in seen_ids or not tk <= ptk:
                continue
            seen_ids.add(sid)
            rows.append({"pep_id": p["id"], "pep_name": p["name"], "pep_birth": p.get("birth_date"),
                         "sanc_id": sid, "matched_name": sname, "sanc_birth": sbd,
                         "birth_match": bool(pd.notna(sbd) and pbd != "nan" and str(sbd)[:10] == pbd),
                         "sanc_dataset": sds})

cands = pd.DataFrame(rows, columns=["pep_id", "pep_name", "pep_birth", "sanc_id",
                                    "matched_name", "sanc_birth", "birth_match", "sanc_dataset"])
cands = cands[~cands["pep_id"].isin(id_hits["id"])].copy()
cands["verdict"] = cands["birth_match"].map(
    {True: "동일인 가능성 높음 (생년월일 일치)", False: "오탐 가능성 (생년월일 불일치·없음)"})

print(f"\n[제재 ID 일치 — 확정] {len(id_hits)}명")
if len(id_hits):
    cols = [c for c in ["name", "birth_date", "sanctions", "dataset_sanc"] if c in id_hits.columns]
    print(id_hits[cols].to_string(index=False))
print(f"\n[이름·별칭 일치 — 후보] {len(cands)}건 (생년월일 일치 {int(cands['birth_match'].sum())}건)")
if len(cands):
    print(cands[["pep_name", "matched_name", "verdict"]].to_string(index=False))

# ───────────────────────────────────────────────
# 3단계: 감시 대상 정하기 (확정 → 생년월일 일치 후보 → 후보 전체)
# ───────────────────────────────────────────────
watch = list(id_hits["name"]) + cands.loc[cands["birth_match"], "pep_name"].tolist()
if not watch:
    watch = cands["pep_name"].tolist()
watch = list(dict.fromkeys(watch + WATCH_EXTRA))

print(f"\n감시 대상 {len(watch)}명")
for n in watch:
    print(" -", n)
if not watch:
    raise SystemExit("감시할 제재 인물이 없어요. WATCH_EXTRA에 이름을 직접 넣어보세요.")


# ───────────────────────────────────────────────
# 4단계: 인물 이름으로 구글 뉴스 RSS 검색 (프랑스어·콩고판)
# ───────────────────────────────────────────────
def short(n):                                        # 후성·부칭 빼고 이름+성
    t = n.split()
    return f"{t[0]} {t[-1]}" if len(t) >= 3 else n


def query_for(n):
    # 콩고 공식 명단은 "성 후성 이름" 순서, 기사는 보통 "이름 성" 순서 → 두 순서 모두 검색
    t = n.split()
    a, b = (t[0], t[-1]) if len(t) >= 2 else (n, "")
    return f'"{b} {a}" OR "{a} {b}"' if b else f'"{a}"'


FEED_PERSON = {}
for n in watch:
    q = quote(query_for(n))
    FEED_PERSON[f"https://news.google.com/rss/search?q={q}&hl=fr&gl=CD&ceid=CD:fr"] = n


def fetch_person_news():
    items = []
    for url, person in FEED_PERSON.items():
        # requests로 받아서(리다이렉트 자동 추적) feedparser에 넘김
        try:
            r = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=True)
        except Exception as e:
            print(f"  ⚠ {person}: 요청 실패 ({e})")
            continue
        feed = feedparser.parse(r.content)
        if r.status_code != 200 or not feed.entries:
            print(f"  ⚠ {person}: 상태 {r.status_code}, 기사 {len(feed.entries)}건, 최종 주소 {r.url[:80]}")
            if "consent" in r.url:
                print("     → 구글 동의(consent) 페이지로 이동됨: 브라우저에서 위 주소를 한 번 열어보세요")
        print(f"  {person}: {len(feed.entries)}건")
        for e in feed.entries:
            items.append({"person": person,
                          "title": e.get("title"),
                          "source": e.get("source", {}).get("title", ""),
                          "published": e.get("published", "")[:25],
                          "url": e.get("link")})
        time.sleep(3)                                # 인물별 요청 간격
    return items


# ───────────────────────────────────────────────
# 5단계: 새 기사만 표시·저장 (LOOP=True면 자동 반복)
# ───────────────────────────────────────────────
OUT = f"{PREFIX}_sanction_news_monitor.csv"
seen = set(pd.read_csv(OUT)["url"]) if os.path.exists(OUT) else set()
first_run = not seen

while True:
    print(f"\n[{datetime.now():%Y-%m-%d %H:%M}] {len(watch)}명 뉴스 확인 중...")
    articles = fetch_person_news()
    new = [a for a in articles if a["url"] and a["url"] not in seen]
    for a in new:
        seen.add(a["url"])
        a["found_at"] = f"{datetime.now():%Y-%m-%d %H:%M}"

    label = "기존 기사(첫 실행)" if first_run else "새 기사"
    print(f"  받은 기사 {len(articles)}건 · {label} {len(new)}건")
    for a in sorted(new, key=lambda x: x["person"])[:30]:
        print(f"  🔔 {a['person']} | {a['title']} ({a['source']}, {a['published'][:16]})")

    if new:
        pd.DataFrame(new).to_csv(OUT, mode="a", index=False,
                                 header=not os.path.exists(OUT), encoding="utf-8-sig")
    first_run = False
    if not LOOP:
        break
    time.sleep(INTERVAL)

# ───────────────────────────────────────────────
# 6단계: 결과 저장 (분석용, 재배포 금지)
# ───────────────────────────────────────────────
peps.to_csv(f"{PREFIX}_peps.csv", index=False, encoding="utf-8-sig")
id_hits.to_csv(f"{PREFIX}_sanction_id_hits.csv", index=False, encoding="utf-8-sig")
cands.to_csv(f"{PREFIX}_sanction_candidates.csv", index=False, encoding="utf-8-sig")

# ───────────────────────────────────────────────
# 7단계: 리포트용 요약
# ───────────────────────────────────────────────
def n_lists(x):
    return len(str(x).split(";")) if pd.notna(x) else 0

summary = []
for _, r in id_hits.iterrows():
    summary.append({"인물": r["name"], "판정": "확정 (ID 일치)",
                    "제재 목록 수": n_lists(r.get("dataset_sanc")), "제재 목록": r.get("dataset_sanc")})
for _, r in cands.iterrows():
    summary.append({"인물": r["pep_name"], "판정": r["verdict"],
                    "제재 목록 수": n_lists(r["sanc_dataset"]), "제재 목록": r["sanc_dataset"]})
summary = pd.DataFrame(summary)

news_all = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["person"])
news_cnt = news_all["person"].value_counts()
if len(summary):
    summary["누적 기사 수"] = summary["인물"].map(news_cnt).fillna(0).astype(int)
    summary.to_csv(f"{PREFIX}_summary.csv", index=False, encoding="utf-8-sig")

print("\n" + "═" * 60)
print(f"요약: {COUNTRY} 의원 {len(peps)}명 중")
print(f"  · 제재 확정 {len(id_hits)}명")
print(f"  · 동일인 가능성 높은 후보 {int(cands['birth_match'].sum())}명")
print(f"  · 오탐으로 판단한 후보 {int((~cands['birth_match']).sum())}명")
if len(summary):
    print(summary[["인물", "판정", "제재 목록 수", "누적 기사 수"]].to_string(index=False))
print("═" * 60)
print("\n✅ 저장 완료")

# ───────────────────────────────────────────────
# 8단계: 인물별 최신 기사 5건 — 원문 헤드라인 + 한글 번역
# ───────────────────────────────────────────────
try:
    from deep_translator import GoogleTranslator
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deep-translator"])
    from deep_translator import GoogleTranslator
from IPython.display import display

pd.set_option("display.max_colwidth", None)
translator = GoogleTranslator(source="auto", target="ko")


def to_korean(text):
    try:
        return translator.translate(str(text))
    except Exception:
        return "(번역 실패)"


def clean_headline(title, source):
    # 구글 뉴스 제목 끝의 " - 언론사명" 제거
    t, s = str(title), str(source)
    return t.rsplit(" - ", 1)[0] if s and t.endswith(s) else t


news = pd.read_csv(OUT)
news["date"] = pd.to_datetime(news["published"], errors="coerce", utc=True).dt.strftime("%Y-%m-%d")
news["headline"] = [clean_headline(t, s) for t, s in zip(news["title"], news["source"].fillna(""))]

top_all = []
for person in watch:
    top = (news[news["person"] == person]
           .sort_values("date", ascending=False)
           .drop_duplicates("headline")
           .head(5)
           .copy())
    top["한글 번역"] = [to_korean(h) for h in top["headline"]]
    top_all.append(top)
    print(f"\n■ {person} — 최신 기사 {len(top)}건")
    display(top[["date", "source", "headline", "한글 번역"]]
            .rename(columns={"date": "날짜", "source": "언론사", "headline": "원문 헤드라인"})
            .reset_index(drop=True))

if top_all:
    pd.concat(top_all).to_csv(f"{PREFIX}_news_top5_ko.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 번역본 저장: drc_news_top5_ko.csv (기계 번역이라 원문과 함께 확인하세요)")

상태 코드: 200

DR Congo 데이터셋 1개
 - DR Congo Members of the National Assembly (cd_national_assembly)
✅ DR Congo Members of the National Assembly: 560행

인물 560명 (예상 약 561명)
제재명단 73,810건

[제재 ID 일치 — 확정] 1명
                        name birth_date  sanctions                                                                                                                                                                                                                                       dataset_sanc
Bitakwira Bihona Hayi Justin 1960-12-05        NaN Belgian Financial Sanctions;EU Consolidated Travel Bans;EU Council Official Journal Sanctioned Entities;EU Financial Sanctions Files (FSF);French National Asset Freezing System;Monaco National Fund Freezing List;Swiss SECO Sanctions/Embargoes

[이름·별칭 일치 — 후보] 2건 (생년월일 일치 1건)
                        pep_name    matched_name              verdict
          Ndjabu Ngabu Floribert Floribert Ngabu 동일인 가능성 높음 (생년월일 일치)
Wagner Tshienda Tshienda Richard  Rich

,날짜,언론사,원문 헤드라인,한글 번역
0,2026-07-11,mediacongo.net,Actualité | Sud-Kivu : L’élu d’Uvira Justin Bitakwira tire la sonnette d’alarme sur l’avancée des rebelles vers Baraka,(번역 실패)
1,2026-06-08,mediacongo.net,Actualité | Justin Bitakwira plaide pour un dialogue entre majorité et opposition afin d’éviter une crise politique,(번역 실패)
2,2026-04-30,mediacongo.net,Actualité | Assemblée nationale : Justin Bitakwira appelle les parlementaires à se concentrer sur la guerre plutôt que sur les querelles politiques,(번역 실패)
3,2026-03-19,ACP | Agence Congolaise de Presse,Faux ! Justin Bitakwira n’est pas contre le changement de la Constitution,(번역 실패)
4,2025-12-30,Pressenza - International Press Agency,[RD Congo] Rumonge : Justin Bitakwira apporte une aide d’urgence à plus de 1000 réfugiés congolais en détresse - Pressenza,(번역 실패)



■ Ndjabu Ngabu Floribert — 최신 기사 5건


,날짜,언론사,원문 헤드라인,한글 번역
0,2023-02-03,DW.com,"Floribert Ndjabu : ""Ils acceptent de déposer les armes""",(번역 실패)
1,2023-01-25,DW.com,"""La situation doit d'abord se calmer"" - Floribert Ndjabu Ngabu",(번역 실패)
2,2022-04-12,RFI,RDC: Thomas Lubanga et Floribert Ndjabu retrouvent la liberté en Ituri,(번역 실패)
3,2022-04-12,Actualite.cd,"RDC: Thomas Lubanga, Floribert Ndjabu et deux officiers de l’armée finalement libérés par les miliciens CODECO",(번역 실패)
4,2022-03-11,Actualite.cd,"RDC: Thomas Lubanga, Germain Katanga, Floribert Ndjabu et 5 autres membres de leur délégation toujours en otage",(번역 실패)



✅ 번역본 저장: drc_news_top5_ko.csv (기계 번역이라 원문과 함께 확인하세요)


# DR Congo PEP 제재 스크리닝 및 뉴스 헤드라인 수집 리포트

## 1. 목표 질문

특정 국가(콩고민주공화국, DR Congo)의 PEP 명단을 발췌한 뒤, 두 가지 질문에 답하는 것을 목표로 했다. 첫째, 명단의 인물 중 **제재 대상**이 있는가. 둘째, 제재 대상 인물에 관한 **뉴스 헤드라인**을 자동으로 찾을 수 있는가. 이는 금융기관 KYC 절차의 제재 스크리닝과 부정적 언론 검색(adverse media screening)을 축소해 재현한 것이며, 앞선 알바니아 분석에서 드러난 문제를 개선해 적용했다.

## 2. 데이터와 방법

PEP 명단은 OpenSanctions의 `/pep/` 페이지(정적 페이지)에서 BeautifulSoup으로 DR Congo 데이터셋 링크를 찾아 공식 CSV로 받았다. 제재 여부는 OpenSanctions의 **제재명단 통합 파일**(73,810건)과 대조했다. robots.txt가 금지한 `/search`, `/entities` 경로는 사용하지 않았다.

알바니아 분석의 교훈을 반영해 대조 방식을 세 가지로 개선했다. 첫째, 공통 ID가 일치하면 "확정"으로 분류했다. 둘째, 콩고 이름은 "성 + 후성(postnom) + 이름" 순서처럼 어순이 일정하지 않으므로, 제재명단 이름이나 별칭(aliases)의 **단어 묶음이 PEP 이름에 모두 포함되는지**로 후보를 찾았다(어순 무관). 셋째, 후보는 **생년월일 일치 여부**로 동일인과 오탐을 구분했다.

뉴스는 처음에 GDELT API를 사용했으나, 인물별 반복 요청으로 **429(요청 과다)** 응답이 지수 백오프(30·60·120초) 후에도 해제되지 않았다. 이에 전원 검색 대신 **제재 관련 인물만** 대상으로 좁히고, 소스를 **구글 뉴스 RSS(프랑스어·콩고판)** 로 바꿨다. 기사는 보통 "이름 성" 순서로 쓰므로 두 어순을 모두 검색했다.

## 3. 수행 과정과 결과

### 3.1 명단 발췌

`/pep/` 페이지에서 DR Congo 관련 데이터셋은 1개(**DR Congo Members of the National Assembly**, `cd_national_assembly`)였다. 상태 코드 200으로 정상 수신했고, **인물 560명**을 확보했다(표 기준 561명과 1건 차이, 영향 미미).

### 3.2 질문 1: 제재 대상인가

| 구분 | 인물 | 판정 근거 | 제재 목록 수 |
|---|---|---|---|
| 확정 | Bitakwira Bihona Hayi Justin | OpenSanctions ID 일치 | 7 |
| 동일인 가능성 높음 | Ndjabu Ngabu Floribert | 이름(Floribert Ngabu) + 생년월일 일치 | 15 |
| 오탐 | Wagner Tshienda Tshienda Richard | 이름 단어만 겹침(Richard Wagner), 생년월일 불일치 | 1 |

**Bitakwira Bihona Hayi Justin**은 EU 여행금지·자산동결, 벨기에·프랑스·스위스·모나코 목록에 올라 있어 **유럽 중심의 제재**를 받고 있다.

**Ndjabu Ngabu Floribert**는 OpenSanctions가 아직 ID로 연결하지 않았지만, 생년월일까지 일치하고 **UN 안보리 통합 제재목록**을 비롯해 영국, 호주, 일본, 남아공 등 15개 목록에 올라 있다. 제재 범위가 국제적이라는 점에서 위험도가 더 높은 사례다.

**Wagner Tshienda Tshienda Richard**는 "Richard"와 "Wagner"라는 흔한 단어가 이란 제재명단의 다른 인물과 겹쳐 잡힌 **동명이인 오탐(false positive)** 이다. 생년월일 대조 단계가 없었다면 제재 대상으로 잘못 분류됐을 것이다.

결과적으로 **의원 560명 중 2명(약 0.36%)이 국제 제재 대상**으로 판단된다.

### 3.3 질문 2: 뉴스 헤드라인을 찾을 수 있는가

| 인물 | 수집 기사 수 |
|---|---|
| Bitakwira Bihona Hayi Justin | 74건 |
| Ndjabu Ngabu Floribert | 9건 |

두 인물 모두 기사를 확보했다. 인물별 최신 5건의 원문 헤드라인과 한글 번역(기계 번역)은 아래 코드 셀을 실행하면 `drc_news_top5_ko.csv`에서 불러와 표로 표시된다.